# 📊 Craigslist Car Price Model - Performance Tracking Dashboard

**Author:** [Your Name] - NetID: [Your NetID]  
**Course:** Operations and Information Management - University of Connecticut  
**Project:** A08 Midterm - Enhanced Craigslist LLM + Modeling Challenge

---

## 📝 Overview

This notebook tracks the evolution of our **enhanced car price prediction model** over time. It shows:

1. **Model Performance Metrics**: MAE, RMSE, MAPE, Bias over time
2. **Feature Importance Evolution**: How feature importance changes as we get more data
3. **Partial Dependence Trends**: How the model uses top features

### Key Improvements from Baseline:
- ✅ **Extended Feature Set**: Added color, city, state, zip_code (10 total features)
- ✅ **Hyperparameter Tuning**: Optuna-based optimization (20 trials)
- ✅ **Advanced Models**: Random Forest / Gradient Boosting (vs simple Decision Tree)
- ✅ **Interpretability**: Permutation importance + Partial Dependence Plots

---

## 🔧 Setup: Clone Repository & Install Dependencies

In [ ]:
# Install required packages
!pip install -q pandas numpy matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully!")

## 📥 Clone Your GitHub Repository

In [ ]:
# CHANGE THIS to your GitHub repository URL
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ⚠️ UPDATE THIS!

# Clone the repo (if not already cloned)
repo_name = GITHUB_REPO.split('/')[-1].replace('.git', '')

if not os.path.exists(repo_name):
    !git clone {GITHUB_REPO}
    print(f"✅ Cloned {repo_name}")
else:
    print(f"✅ Repository {repo_name} already exists")
    # Pull latest changes
    !cd {repo_name} && git pull
    print("✅ Pulled latest changes")

## 📂 Load All Prediction Files

In [ ]:
# Path to your results folder
RESULTS_PATH = f"{repo_name}/results"  # Adjust if different
ENHANCED_RESULTS_PATH = f"{repo_name}/enhanced_results"  # For new enhanced model outputs

# Load all prediction CSV files
def load_all_predictions(results_path):
    """
    Load all prediction CSVs and calculate metrics for each timestamp
    """
    pred_files = sorted(glob.glob(f"{results_path}/*-preds.csv"))
    
    if not pred_files:
        print(f"⚠️ No prediction files found in {results_path}")
        return pd.DataFrame()
    
    print(f"📊 Found {len(pred_files)} prediction files")
    
    metrics_list = []
    
    for file_path in pred_files:
        # Extract timestamp from filename (e.g., 2026030723-preds.csv -> 2026030723)
        timestamp_str = os.path.basename(file_path).split('-')[0]
        
        try:
            # Parse timestamp
            timestamp = pd.to_datetime(timestamp_str, format='%Y%m%d%H')
            
            # Load predictions
            df = pd.read_csv(file_path)
            
            # Filter out rows with missing actual or predicted prices
            df_clean = df.dropna(subset=['actual_price', 'pred_price'])
            
            if len(df_clean) == 0:
                continue
            
            # Calculate metrics
            mae = np.mean(np.abs(df_clean['actual_price'] - df_clean['pred_price']))
            rmse = np.sqrt(np.mean((df_clean['actual_price'] - df_clean['pred_price'])**2))
            
            # MAPE (avoid division by zero)
            mape_mask = df_clean['actual_price'] != 0
            if mape_mask.any():
                mape = np.mean(np.abs(
                    (df_clean.loc[mape_mask, 'actual_price'] - df_clean.loc[mape_mask, 'pred_price']) / 
                    df_clean.loc[mape_mask, 'actual_price']
                )) * 100
            else:
                mape = np.nan
            
            # Bias
            bias = np.mean(df_clean['pred_price'] - df_clean['actual_price'])
            
            metrics_list.append({
                'timestamp': timestamp,
                'date': timestamp.date(),
                'hour': timestamp.hour,
                'n_predictions': len(df_clean),
                'mae': mae,
                'rmse': rmse,
                'mape': mape,
                'bias': bias,
                'file': os.path.basename(file_path)
            })
            
        except Exception as e:
            print(f"⚠️ Error processing {file_path}: {e}")
            continue
    
    metrics_df = pd.DataFrame(metrics_list)
    return metrics_df

# Load predictions
metrics_df = load_all_predictions(RESULTS_PATH)

if not metrics_df.empty:
    print(f"\n✅ Loaded {len(metrics_df)} successful prediction runs")
    print(f"📅 Date range: {metrics_df['date'].min()} to {metrics_df['date'].max()}")
    print(f"\n📊 Summary Statistics:")
    print(metrics_df[['mae', 'rmse', 'mape', 'bias']].describe())
else:
    print("❌ No valid prediction data found")

## 📊 Dashboard 1: Model Performance Over Time

In [ ]:
if not metrics_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('🚗 Model Performance Metrics Over Time', fontsize=16, fontweight='bold')
    
    # 1. MAE over time
    axes[0, 0].plot(metrics_df['timestamp'], metrics_df['mae'], marker='o', linewidth=2, markersize=4, color='#2E86AB')
    axes[0, 0].axhline(metrics_df['mae'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: ${metrics_df["mae"].mean():.0f}')
    axes[0, 0].set_title('Mean Absolute Error (MAE)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('MAE ($)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. RMSE over time
    axes[0, 1].plot(metrics_df['timestamp'], metrics_df['rmse'], marker='s', linewidth=2, markersize=4, color='#A23B72')
    axes[0, 1].axhline(metrics_df['rmse'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: ${metrics_df["rmse"].mean():.0f}')
    axes[0, 1].set_title('Root Mean Squared Error (RMSE)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Date')
    axes[0, 1].set_ylabel('RMSE ($)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. MAPE over time
    axes[1, 0].plot(metrics_df['timestamp'], metrics_df['mape'], marker='^', linewidth=2, markersize=4, color='#F18F01')
    axes[1, 0].axhline(metrics_df['mape'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {metrics_df["mape"].mean():.1f}%')
    axes[1, 0].set_title('Mean Absolute Percentage Error (MAPE)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Date')
    axes[1, 0].set_ylabel('MAPE (%)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 4. Bias over time
    axes[1, 1].plot(metrics_df['timestamp'], metrics_df['bias'], marker='D', linewidth=2, markersize=4, color='#6A994E')
    axes[1, 1].axhline(0, color='black', linestyle='-', alpha=0.5, linewidth=1)
    axes[1, 1].axhline(metrics_df['bias'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: ${metrics_df["bias"].mean():.0f}')
    axes[1, 1].set_title('Model Bias (Over/Under Prediction)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('Bias ($) [+ = Over-prediction]')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print insights
    print("\n🔍 KEY INSIGHTS:")
    print(f"Average MAE: ${metrics_df['mae'].mean():.2f} (Lower is better)")
    print(f"Average RMSE: ${metrics_df['rmse'].mean():.2f} (Lower is better)")
    print(f"Average MAPE: {metrics_df['mape'].mean():.2f}% (Lower is better)")
    print(f"Average Bias: ${metrics_df['bias'].mean():.2f} (0 = no systematic bias)")
    
    if metrics_df['bias'].mean() > 0:
        print("   → Model tends to OVER-predict prices")
    elif metrics_df['bias'].mean() < 0:
        print("   → Model tends to UNDER-predict prices")
    else:
        print("   → Model is well-calibrated (no systematic bias)")
else:
    print("❌ No data to plot")

## 📊 Dashboard 2: Feature Importance Evolution

In [ ]:
# Load feature importance files (if using enhanced training)
def load_all_importance(results_path):
    """
    Load all importance CSV files and track evolution
    """
    importance_files = sorted(glob.glob(f"{results_path}/*-importance.csv"))
    
    if not importance_files:
        print(f"⚠️ No importance files found in {results_path}")
        return {}
    
    print(f"📊 Found {len(importance_files)} importance files")
    
    importance_over_time = {}
    
    for file_path in importance_files:
        timestamp_str = os.path.basename(file_path).split('-')[0]
        
        try:
            timestamp = pd.to_datetime(timestamp_str, format='%Y%m%d%H')
            df = pd.read_csv(file_path)
            importance_over_time[timestamp] = df
        except Exception as e:
            print(f"⚠️ Error loading {file_path}: {e}")
            continue
    
    return importance_over_time

# Try to load from enhanced results first, then fall back to regular results
importance_data = load_all_importance(ENHANCED_RESULTS_PATH)
if not importance_data:
    importance_data = load_all_importance(RESULTS_PATH)

if importance_data:
    # Plot feature importance evolution
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Get the latest importance data
    latest_timestamp = max(importance_data.keys())
    latest_importance = importance_data[latest_timestamp]
    
    # Plot horizontal bar chart
    colors = plt.cm.viridis(np.linspace(0, 1, len(latest_importance)))
    bars = ax.barh(latest_importance['feature'], latest_importance['importance_mean'], color=colors)
    
    # Add error bars
    ax.errorbar(
        latest_importance['importance_mean'], 
        latest_importance['feature'],
        xerr=latest_importance['importance_std'],
        fmt='none',
        ecolor='black',
        alpha=0.5,
        capsize=3
    )
    
    ax.set_xlabel('Permutation Importance (Mean ± Std)', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    ax.set_title(f'Feature Importance (Latest: {latest_timestamp.strftime("%Y-%m-%d %H:00")})', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("\n🔍 TOP 5 MOST IMPORTANT FEATURES:")
    print(latest_importance.head()[['feature', 'importance_mean', 'importance_std']].to_string(index=False))
else:
    print("⚠️ No feature importance data available yet. Run the enhanced training model to generate importance files.")

## 📊 Dashboard 3: Model Improvement Trends

In [ ]:
if not metrics_df.empty:
    # Calculate rolling averages (7-day window)
    metrics_df = metrics_df.sort_values('timestamp')
    metrics_df['mae_rolling'] = metrics_df['mae'].rolling(window=min(7, len(metrics_df)), min_periods=1).mean()
    metrics_df['rmse_rolling'] = metrics_df['rmse'].rolling(window=min(7, len(metrics_df)), min_periods=1).mean()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('📈 Model Improvement Trends (Rolling Averages)', fontsize=16, fontweight='bold')
    
    # MAE trend with rolling average
    axes[0].plot(metrics_df['timestamp'], metrics_df['mae'], alpha=0.3, label='Raw MAE', color='#2E86AB')
    axes[0].plot(metrics_df['timestamp'], metrics_df['mae_rolling'], linewidth=3, label='7-period Moving Avg', color='#2E86AB')
    axes[0].set_title('MAE Trend Analysis', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('MAE ($)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Number of predictions over time
    axes[1].bar(metrics_df['timestamp'], metrics_df['n_predictions'], color='#6A994E', alpha=0.7)
    axes[1].set_title('Dataset Growth (Predictions per Run)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Number of Predictions')
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate trend
    if len(metrics_df) > 1:
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(
            range(len(metrics_df)), metrics_df['mae']
        )
        
        print("\n🔍 TREND ANALYSIS:")
        if slope < 0:
            print(f"✅ Model is IMPROVING over time (MAE decreasing by ${abs(slope):.2f} per run)")
        elif slope > 0:
            print(f"⚠️ Model performance is DEGRADING over time (MAE increasing by ${slope:.2f} per run)")
        else:
            print("→ Model performance is STABLE")
        
        print(f"R² = {r_value**2:.4f} (trend strength)")
        print(f"Total predictions made: {metrics_df['n_predictions'].sum()}")
else:
    print("❌ No data to analyze trends")

## 🎯 Summary & Key Findings

In [ ]:
if not metrics_df.empty:
    print("="*70)
    print("📊 CRAIGSLIST CAR PRICE MODEL - PERFORMANCE SUMMARY")
    print("="*70)
    print(f"\n📅 Analysis Period: {metrics_df['date'].min()} to {metrics_df['date'].max()}")
    print(f"📈 Total Runs: {len(metrics_df)}")
    print(f"🎯 Total Predictions: {metrics_df['n_predictions'].sum()}")
    
    print("\n🎯 CURRENT MODEL PERFORMANCE:")
    latest = metrics_df.iloc[-1]
    print(f"   MAE:  ${latest['mae']:.2f}")
    print(f"   RMSE: ${latest['rmse']:.2f}")
    print(f"   MAPE: {latest['mape']:.2f}%")
    print(f"   Bias: ${latest['bias']:.2f}")
    
    print("\n📊 OVERALL STATISTICS:")
    print(f"   Average MAE:  ${metrics_df['mae'].mean():.2f} ± ${metrics_df['mae'].std():.2f}")
    print(f"   Average RMSE: ${metrics_df['rmse'].mean():.2f} ± ${metrics_df['rmse'].std():.2f}")
    print(f"   Average MAPE: {metrics_df['mape'].mean():.2f}% ± {metrics_df['mape'].std():.2f}%")
    
    print("\n🏆 BEST PERFORMANCE ACHIEVED:")
    best_idx = metrics_df['mae'].idxmin()
    best = metrics_df.loc[best_idx]
    print(f"   Date: {best['timestamp'].strftime('%Y-%m-%d %H:00')}")
    print(f"   MAE:  ${best['mae']:.2f}")
    print(f"   RMSE: ${best['rmse']:.2f}")
    print(f"   MAPE: {best['mape']:.2f}%")
    
    if importance_data:
        print("\n🔑 TOP 3 MOST IMPORTANT FEATURES:")
        top_3 = latest_importance.head(3)
        for idx, row in top_3.iterrows():
            print(f"   {idx+1}. {row['feature']}: {row['importance_mean']:.4f}")
    
    print("\n" + "="*70)
    print("✅ Dashboard complete! Model is ready for production.")
    print("="*70)
else:
    print("❌ No data available for summary")

---

## 💡 Insights & What I Learned

**Key Observations:**
1. [Add your observation about model performance trends]
2. [Add your observation about feature importance]
3. [Add your observation about data quality or patterns]

**Surprises:**
- [Something unexpected you discovered during the project]

**Future Improvements:**
- [Ideas for making the model even better]

---

**End of Dashboard** 🎉